# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: The Freshness Multiplier

The paper reports a large increase in impressions for refreshed mature pages and describes freshness as one of the strongest observed signals in the dataset.

**Methodology question:** Where does the outcome label come from? I would want to confirm exactly how a page is classified as “refreshed” and how the later impression outcome is measured. I would also ask whether the comparison controls for page age, position, and other differences between refreshed and non-refreshed pages.

**Validation question:** Does the validation design support the strength of the claim? The paper describes this as a pattern study and notes that the results do not prove cause and effect. Therefore, the finding is useful as an observed association, but I would not interpret the measured impression increase as proof that refreshing itself caused the increase.

### Finding 2: The CTR Cliff

The paper reports that CTR changes substantially across search-position tiers and uses position as an important part of its analysis.

**Methodology question:** Where does the label or outcome come from? I would want to confirm that CTR is calculated from observed Search Console clicks and impressions for the same measurement window, and that pages with very different impression volumes are handled consistently.

**Validation question:** Does the validation design carry the claim? Position and CTR are naturally related, so I would want to check whether the comparison controls for differences in page mix and search visibility. The paper's methodology says its headline findings are based mainly on direct aggregate comparisons, so the result should be read as an observed portfolio pattern rather than proof that position alone causes the CTR change.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# Section 2 — Honest client-grouped validation
# Recreate the Week-5 modeling dataset

import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

# Load Hugging Face data
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

query = f"""
WITH base AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,
        scroll_events
    FROM {rel}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-30'
),

current_data AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,
        scroll_events,

        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS ctr

    FROM base
),

next_day AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS next_ctr

    FROM base
)

SELECT
    c.*,
    n.next_ctr,

    CASE
        WHEN c.ctr IS NOT NULL
             AND n.next_ctr IS NOT NULL
             AND n.next_ctr > c.ctr
        THEN 1
        ELSE 0
    END AS target

FROM current_data c

LEFT JOIN next_day n
    ON c.client_hash_id = n.client_hash_id
    AND c.content_hash_id = n.content_hash_id
    AND n.report_date = c.report_date + INTERVAL 1 DAY

WHERE c.gsc_impressions > 0
  AND c.gsc_avg_position > 0
  AND n.next_ctr IS NOT NULL
  AND c.gsc_impressions >= 5
"""

print("Running DuckDB query...")

training_df = con.sql(query).df()

print("Original dataset shape:", training_df.shape)


# Same Week-5 sampling
MAX_ROWS = 300000

if len(training_df) > MAX_ROWS:
    training_df = (
        training_df
        .sample(
            n=MAX_ROWS,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )

print("Final ML dataset shape:", training_df.shape)


# Same Week-5 features
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

X = training_df[feature_cols].copy()
y = training_df["target"].astype(int)
groups = training_df["client_hash_id"]


# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()


# Check client separation
train_clients = set(
    training_df.iloc[train_idx]["client_hash_id"]
)

test_clients = set(
    training_df.iloc[test_idx]["client_hash_id"]
)

client_overlap = train_clients.intersection(test_clients)

print("\n--- Honest Split ---")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))


# Logistic Regression
honest_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

honest_model.fit(X_train, y_train)

honest_probability = (
    honest_model.predict_proba(X_test)[:, 1]
)

honest_roc_auc = roc_auc_score(
    y_test,
    honest_probability
)

print("\n--- Honest Validation Result ---")
print(
    "Grouped-test ROC AUC:",
    round(honest_roc_auc, 4)
)

Running DuckDB query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Original dataset shape: (2409018, 20)
Final ML dataset shape: (300000, 20)

--- Honest Split ---
Training rows: 282244
Test rows: 17756
Training clients: 35
Test clients: 9
Client overlap: 0

--- Honest Validation Result ---
Grouped-test ROC AUC: 0.7426


### Validation comparison

The Week-5 model was evaluated using a client-grouped 80/20 split. This keeps complete clients in either training or testing and prevents the same client from appearing in both sets.

The earlier Week-5 evaluation used this grouped split and produced a ROC AUC of approximately **0.7471**. The ranking evaluation also measured Precision@20 of **0.60** and Precision@50 of **0.60**.

The honest validation design therefore keeps the evaluation focused on unseen clients. The results are interpreted as measured performance on this held-out sample, not as proof that the model will perform identically on every future client.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Leakage audit for the final Week-5 feature set

leakage_keywords = [
    "target",
    "next_",
    "future",
    "trend",
    "label",
    "outcome"
]

leakage_check = []

for feature in feature_cols:
    matched_terms = [
        word for word in leakage_keywords
        if word.lower() in feature.lower()
    ]

    leakage_check.append({
        "feature": feature,
        "possible_leakage_keyword": (
            ", ".join(matched_terms)
            if matched_terms else "None"
        )
    })

leakage_audit = pd.DataFrame(leakage_check)

display(leakage_audit)

print("\nExcluded future/label-derived columns:")
print([
    col for col in training_df.columns
    if any(word in col.lower() for word in leakage_keywords)
])

,feature,possible_leakage_keyword
0,gsc_impressions,None
1,gsc_clicks,None
2,ctr,None
3,gsc_avg_position,None
4,ga4_pageviews,None
5,ga4_sessions,None
6,ga4_engaged_sessions,None
7,ga4_total_engagement_sec,None
8,sessions_organic,None
9,sessions_direct,None



Excluded future/label-derived columns:
['next_ctr', 'target']


### Leakage audit

The final model features were checked for future-looking and label-derived fields.

The target and `next_ctr` are excluded from the feature set. Trend-related fields were also not used as model features. The model uses observed search-performance and engagement signals such as impressions, clicks, CTR, average position, sessions, and scroll events.

The client identifier is used only for grouped splitting and is not used as a model feature.

This audit supports the conclusion that the final feature set does not intentionally include the future target. However, the features are still observational measurements, so their relationship with the target should be interpreted as directional rather than causal.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# Real failure examples from the honest grouped test set

error_examples = training_df.iloc[test_idx][
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "next_ctr",
        "target"
    ]
].copy()

error_examples["model_probability"] = honest_probability

error_examples["prediction"] = (
    error_examples["model_probability"] >= 0.5
).astype(int)

error_examples["error_type"] = np.select(
    [
        (error_examples["target"] == 1) &
        (error_examples["prediction"] == 0),

        (error_examples["target"] == 0) &
        (error_examples["prediction"] == 1)
    ],
    [
        "False Negative",
        "False Positive"
    ],
    default="Correct"
)

print("Error counts:")
display(
    error_examples["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

print("\nFalse Positive examples:")
display(
    error_examples[
        error_examples["error_type"] == "False Positive"
    ].sort_values(
        "model_probability",
        ascending=False
    ).head(5)
)

print("\nFalse Negative examples:")
display(
    error_examples[
        error_examples["error_type"] == "False Negative"
    ].sort_values(
        "model_probability",
        ascending=True
    ).head(5)
)

Error counts:


,error_type,count
0,Correct,15433
1,False Positive,1311
2,False Negative,1012



False Positive examples:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,next_ctr,target,model_probability,prediction,error_type
69242,2026-03-25,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,3633,1,0.000275,7.170383,0.000000,0,1.000000,1,False Positive
159102,2026-03-04,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,3370,0,0.000000,8.021068,0.000000,0,0.999999,1,False Positive
220154,2026-03-08,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,3327,0,0.000000,7.947099,0.000000,0,0.999999,1,False Positive
32072,2026-03-28,client_3f0ce4d44fe94f3d,content_23d3ffd9b93c51b1,1194,2,0.001675,7.992462,0.000611,0,0.988269,1,False Positive
235415,2026-03-25,client_3f0ce4d44fe94f3d,content_1807a69fb828e30d,1158,3,0.002591,6.106218,0.000985,0,0.985697,1,False Positive



False Negative examples:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,next_ctr,target,model_probability,prediction,error_type
123124,2026-03-22,client_f623b01661d4bfe4,content_b8eb55e9605771a2,8,0,0.0,86.125,0.066667,1,0.040780,0,False Negative
171268,2026-03-16,client_f623b01661d4bfe4,content_fd499895590565e9,17,0,0.0,80.000,0.058824,1,0.051940,0,False Negative
203351,2026-03-29,client_2094c6eb080311d5,content_2ece6849c7b26814,5,0,0.0,76.200,0.125000,1,0.056005,0,False Negative
91789,2026-03-28,client_f623b01661d4bfe4,content_f6b05c2d4a4f89f8,16,0,0.0,68.375,0.041667,1,0.075655,0,False Negative
204297,2026-03-11,client_2094c6eb080311d5,content_b678de5511c085dd,10,0,0.0,66.400,0.090909,1,0.078677,0,False Negative


### Claim rewrite

**Earlier claim:**

“The Logistic Regression model significantly outperforms the Week-4 baseline and is a better way to identify CTR improvement opportunities.”

**Safer claim:**

“On the held-out client-grouped test sample, the Logistic Regression model showed higher measured Precision@20 and Precision@50 than the Week-4 rule-based baseline. This is an observed result on this evaluation sample and suggests that the model may provide useful decision-support for prioritizing content review. It does not prove that the model will outperform the baseline for every future client or that the model causes CTR improvement.”


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.